<a href="https://colab.research.google.com/github/maheshkumar30/DataScience/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import re
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SpatialDropout1D, LSTM, Dense
from tensorflow.keras.utils import to_categorical

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [8]:
# Load CSV file
df = pd.read_csv("Tweets.csv")


In [9]:
df.columns

Index(['textID', 'text', 'selected_text', 'sentiment'], dtype='object')

In [11]:
# Select required columns
df = df[['textID', 'sentiment', 'text']]

print(df.head())
print(df.shape)

       textID sentiment                                               text
0  cb774db0d1   neutral                I`d have responded, if I were going
1  549e992a42  negative      Sooo SAD I will miss you here in San Diego!!!
2  088c60f138  negative                          my boss is bullying me...
3  9642c003ef  negative                     what interview! leave me alone
4  358bd9e861  negative   Sons of ****, why couldn`t they put them on t...
(27481, 3)


In [15]:
print(df['text'].isnull().sum())
df['text'] = df['text'].fillna('')


1


/tmp/ipykernel_4995/2446024011.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].fillna('')


#**Text Preprocessing**

In [16]:

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions and hashtags
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0                I`d have responded, if I were going   
1      Sooo SAD I will miss you here in San Diego!!!   
2                          my boss is bullying me...   
3                     what interview! leave me alone   
4   Sons of ****, why couldn`t they put them on t...   

                                 clean_text  
0                        id responded going  
1                   sooo sad miss san diego  
2                             boss bullying  
3                     interview leave alone  
4  sons couldnt put releases already bought  


#**Encode Sentiment Labels**

Assume sentiments are:

* Positive
* Negative
* Neutral

In [17]:
encoder = LabelEncoder()

df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

y = to_categorical(df['sentiment_encoded'])

#**Tokenization and Padding**

In [18]:
max_words = 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])

X = pad_sequences(X, maxlen=max_len)

print(X.shape)

(27481, 50)


#**Train-Test Split**

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#**Build Deep Learning Model**

In [20]:
model = Sequential()

model.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

model.add(SpatialDropout1D(0.3))

model.add(
    LSTM(
        128,
        dropout=0.2,
        recurrent_dropout=0.2
    )
)

model.add(Dense(3, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#**Compile Model**

In [21]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

#**Train Model**

In [22]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 34s 111ms/step - accuracy: 0.5870 - loss: 0.8817 - val_accuracy: 0.6766 - val_loss: 0.7492
Epoch 2/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 34s 124ms/step - accuracy: 0.7412 - loss: 0.6375 - val_accuracy: 0.6914 - val_loss: 0.7466
Epoch 3/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 32s 117ms/step - accuracy: 0.7968 - loss: 0.5228 - val_accuracy: 0.6807 - val_loss: 0.7727
Epoch 4/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 33s 120ms/step - accuracy: 0.8298 - loss: 0.4511 - val_accuracy: 0.6730 - val_loss: 0.8383
Epoch 5/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 32s 118ms/step - accuracy: 0.8540 - loss: 0.3944 - val_accuracy: 0.6693 - val_loss: 0.9126
Epoch 6/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 42s 122ms/step - accuracy: 0.8732 - loss: 0.3529 - val_accuracy: 0.6607 - val_loss: 0.9660
Epoch 7/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 38s 110ms/step - accuracy: 0.8867 - loss: 0.3191 - val_accuracy: 0.6575 - val_loss: 1.0034
Epoch 8/10
275/275 ━━━━━━━━━━━━━━━━━━━━ 42s 114ms/step - accuracy: 0.8993 - loss: 0

#**Evaluate Model**

In [23]:
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

Test Loss : 1.2252259254455566
Test Accuracy : 0.6485355496406555
